大規模言語モデル入門Ⅱの10章を参考に評価用スクリプトの作成を行う。(とりあえず本の写経から)

### 10.2.3 llm-jp-evalで使用される評価指標
- 完全一致率(exact match ratio): 正解テキストと予測テキストが完全一致しているか否かを比較する指標。llm-jp-evalでは自然言語推論、多肢選択式質問応答、数学的推論で使用。

$\text{完全一致率}=\frac{\text{正解事例と予測事例の一致数}}{事例数}$

In [1]:
def calc_exact_match_ratio(trues: list[str], preds: list[str]) -> float:
    """
    完全一致率を計算する関数
    Args:
        trues (list[str]): 正解テキストのリスト
        preds (list[str]): 予測テキストのリスト
    """
    # どちらかの事例がなければ0
    if len(trues) == 0 or len(preds) == 0:
        return 0
    
    # 正解テキストと予測テキストが一致していれば1、そうでなければ0
    num_exact_match = sum(
        1 if t == p else 0 for t, p in zip(trues, preds)
    )
    return num_exact_match / len(trues)

# 正解テキスト列
trues = ["entailment", "entailment", "contradiction"]
# 予測テキスト列
preds = ["entailment", "entailmen", "neutral"]
# 完全一致率を算出する
print("完全一致率:", calc_exact_match_ratio(trues, preds))

完全一致率: 0.3333333333333333


- 文字ベースF値: 正解テキストと予測テキストの文字の一致に基づいた適合率と再現率の調和平均。質問応答と機械読解で使用。

$\text{文字ベース適合率}=\frac{正解テキストと予測テキストで一致した文字数}{予測テキストの文字数}$

$\text{文字ベース再現率}=\frac{正解テキストと予測テキストで一致した文字数}{正解テキストの文字数}$

$\text{文字ベースF値}=\frac{2\cdot \text{文字ベース適合率}\cdot \text{文字ベース再現率}}{\text{文字ベース適合率}+\text{文字ベース再現率}}$

In [4]:
from collections import Counter

def calc_char_f1(trues: list[str], preds: list[str]) -> float:
    """
    文字ベースF値を計算する関数
    Args:
        trues (list[str]): 正解テキストのリスト
        preds (list[str]): 予測テキストのリスト
    """
    # どちらかの事例がなければ0
    if len(trues) == 0 or len(preds) == 0:
        return 0
    
    char_f1_scores = []
    # 各事例ごとに文字ベースF値を算出する
    for t, p in zip(trues, preds):
        # 正解テキストと予測テキストのどちらかの文字がない場合
        if len(t) == 0 or len(p) == 0:
            # 別に0でよくない？
            char_f1_scores.append(float(t==p))
            break
        # 正解テキストと予測テキストの一致している文字を算出する
        common= Counter(list(t)) & Counter(list(p))
        num_same = sum(common.values())
        # 適合率の算出
        precision = num_same / len(p)
        # 再現率の算出
        recall = num_same / len(t)
        # F値の算出
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        char_f1_scores.append(f1_score)

    return sum(char_f1_scores) / len(char_f1_scores)

# 正解のテキスト列
trues = ["夏目漱石"]
# 予測のテキスト列
preds = ["夏目 漱石"]
# 文字ベースF値を算出する
print("文字ベースF値:", calc_char_f1(trues, preds))

文字ベースF値: 0.888888888888889


- 集合ベースF値: 正解が集合で与えられるタスクの性能評価に使う指標。正解集合と予測集合でF1スコアを算出して計算。主にエンティティ極性分析と基礎解析で使用。
(集合: 1つの設問に対して複数の回答テキストが存在する際の回答テキスト集合)

$\text{集合ベース適合率}=\frac{正解集合と予測集合で一致した要素数}{予測テキストの要素数}$

$\text{集合ベース再現率}=\frac{正解集合と予測集合で一致した要素数}{正解テキストの要素数}$

$\text{集合ベースF値}=\frac{2\cdot \text{集合ベース適合率}\cdot \text{集合ベース再現率}}{\text{集合ベース適合率}+\text{集合ベース再現率}}$

In [6]:
def calc_set_f1(trues: list[set[str]], preds: list[set[str]]) -> float:
    """
    集合ベースF値を計算する関数
    Args:
        trues (list[set[str]]): 正解テキスト集合のリスト
        preds (list[set[str]]): 予測テキスト集合のリスト
    """
    # どちらかの事例がなければ0
    if len(trues) == 0 or len(preds) == 0:
        return 0
    
    set_f1_scores = []
    # 事例単位で処理
    for t, p in zip(trues, preds):
        # 正解データを行ごとに分割し、集合にする
        split_t = {x.strip() for x in t.split("\n")}
        # 予測データを行ごとに分割し、集合にする
        split_p = {x.strip() for x in p.split("\n")}
        # 適合率の算出
        precision = sum(1 if y in split_t else 0 for y in split_p) / len(split_p)
        # 再現率の算出
        recall = sum(1 if y in split_p else 0 for y in split_t) / len(split_t)
        # F値の算出
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        set_f1_scores.append(f1_score)
    
    return sum(set_f1_scores) / len(set_f1_scores)

# 正解集合
trues = [
    "営業収益 positive\n"
    "純営業収益 positive\n"
    "経常利益 positive\n"
    "当期純利益 positive"
]
# 予測集合
preds = ["営業収益 positive\n純営業収益 negative\n経常利益 positive"]
# 集合ベースF値を算出する
print("集合ベースF値:", calc_set_f1(trues, preds))

集合ベースF値: 0.5714285714285715


- 相関係数: ピアソンの積率相関係数とスピアマンの順位相関係数がある。実数値でないと算出できないので注意。

In [8]:
import math
from scipy.stats import pearsonr, spearmanr

def calc_pearsonr(trues: list[float], preds: list[float]) -> float:
    """
    ピアソンの積率相関係数を計算する関数
    Args:
        trues (list[float]): 正解値のリスト
        preds (list[float]): 予測値のリスト
    """
    scores = pearsonr(
        list(map(float, trues)), list(map(float, preds))
    ).statistic

    return 0.0 if math.isnan(scores) else scores

def calc_spearmanr(trues: list[float], preds: list[float]) -> float:
    """
    スピアマンの順位相関係数を計算する関数
    Args:
        trues (list[float]): 正解値のリスト
        preds (list[float]): 予測値のリスト
    """
    scores = spearmanr(
        list(map(float, trues)), list(map(float, preds))
    ).statistic

    return 0.0 if math.isnan(scores) else scores

trues = ["1.2", "2.2", "3.3", "4.9"]
preds = ["1.1", "4.1", "4.0", "5.0"]
print("ピアソンの積率相関係数:", calc_pearsonr(trues, preds))
print("スピアマンの順位相関係数:", calc_spearmanr(trues, preds))

ピアソンの積率相関係数: 0.8514063149390616
スピアマンの順位相関係数: 0.7999999999999999


LLM-jp-3.1-1.8Bと適当なデータセットで検証。

とりあえずpipeline経由でやってみる

In [9]:
from transformers import pipeline
model_name = "llm-jp/llm-jp-3.1-1.8b"
pipe = pipeline("text-generation", model=model_name)

# Load model directly
#from transformers import AutoTokenizer, AutoModelForCausalLM

#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00001.safetensors:   0%|          | 0.00/3.74G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

Device set to use cuda:0
